# 01 读取热门话题数据

为收集热门话题，我们使用了一个开源 GitHub 仓库，上面有实时更新并存储的微博热搜词条数据。

每个词条的数据包括：

- 日期
- 词条内容
- 类型
- 点击量

我们提取出2025年的所有热搜词条，并将其整理并保存到 `weibo_trending_data.json` 文件中。

In [ ]:
import json
from typing import Dict, List

file_path = r"..\data\raw\weibo_trending\weibo_trending_data.json"
def read_data(file_path: str) -> List[Dict]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

trendings = read_data(file_path)

len(trendings), trendings[:5]

In [ ]:
from collections import Counter

type_counter = Counter(item["type"].strip() for item in trendings
                       if len(item["type"]) < 10)

print("词条数最多的前20个类型：")
for type_name, count in type_counter.most_common(20):
    print(f"  {type_name}: {count:,} 条")

In [ ]:
import random

selected_types = ["社会", "时事", "财经", "互联网", "科普", "情感"]

selected_trendings = [item for item in trendings if item["type"] in selected_types]

print(f"筛选后词条数: {len(selected_trendings)}")
print(f"占比: {len(selected_trendings) / len(trendings) * 100:.1f}%")
print("筛选词条示例：")

for item in random.sample(selected_trendings, 5):
    print(f" {item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")

In [ ]:
import numpy as np

# 提取所有点击量
clicks = [item["clicks"] for item in selected_trendings]

ratio = 99

threshold = np.percentile(clicks, ratio)

print(f"总词条数: {len(selected_trendings)}")
print(f"点击量{ratio}%分位数: {threshold:,.0f}")
print(f"点击量范围: {min(clicks):,.0f} - {max(clicks):,.0f}")
# print(f"平均点击量: {np.mean(clicks):,.0f}")
# print(f"中位数点击量: {np.median(clicks):,.0f}")

top_1_percent_trendings = [item for item in selected_trendings 
                            if item["clicks"] >= threshold]

# 按点击量降序排序
top_1_percent_trendings.sort(key=lambda x: x["clicks"], reverse=True)

print(f"\n筛选后词条数: {len(top_1_percent_trendings)}")
print(f"占比: {len(top_1_percent_trendings) / len(trendings) * 100:.1f}%")

In [ ]:
print("热搜词条示例：")
for item in random.sample(top_1_percent_trendings, 5):
    print(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")


In [ ]:
# 查看筛选后的数据统计
print("=" * 60)
print("筛选后数据统计")
print("=" * 60)
print(f"总词条数: {len(top_1_percent_trendings):,}")
print(f"点击量阈值: {threshold:,.0f} 次")
print(f"最高点击量: {top_1_percent_trendings[0]['clicks']:,} 次")
print(f"最低点击量: {top_1_percent_trendings[-1]['clicks']:,} 次")
print(f"平均点击量: {np.mean([item['clicks'] for item in top_1_percent_trendings]):,.0f} 次")

# 按类型统计
from collections import Counter
type_dist = Counter(item['type'] for item in top_1_percent_trendings)
print(f"\n类型分布:")
for type_name, count in type_dist.most_common():
    percentage = count / len(top_1_percent_trendings) * 100
    print(f"  {type_name}: {count:,} 条 ({percentage:.1f}%)")

In [ ]:
with open("top_1_percent_trendings_detail.txt", 'w', encoding='utf-8') as f:
    for item in top_1_percent_trendings:
        f.write(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,}\n")

trending_names = set([item['name'] for item in top_1_percent_trendings])

with open("top_1_percent_trendings.txt", 'w', encoding='utf-8') as f:
    for name in trending_names:
        f.write(f"{name}\n")